<a href="https://colab.research.google.com/github/aymuos/endgame/blob/main/check_batch_dynamic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Batch Dynamics Check — Single City

This notebook loads a single city's delivery CSV, sorts by `delivery_user_id` and `receipt_time`, computes batch features (batch size, dispatch rank, batch id) and checks for `dynamic pickup` events: where a courier receives a new batch while some orders from the previous batch remain undelivered (sign_time > next_batch_receipt_time).

Edit `CITY` and `CITY_CSV_PATHS` below to point to your local files if needed.

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# Imports and config
import os
import sys
from datetime import datetime
import numpy as np
import pandas as pd

# Select city to analyse (case-sensitive, e.g. 'Shanghai')
CITY = 'Shanghai'

# Candidate paths (edit if your files live elsewhere)
CITY_CSV_PATHS = [
    '/content/drive/MyDrive/ml/PROCESSED/matched/city_divided/shanghai_data.csv',
]

def find_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

CITY_CSV = find_existing_path(CITY_CSV_PATHS)
if CITY_CSV is None:
    print('No candidate CSV found. Please update CITY_CSV_PATHS to point to your file.')
    sys.exit(1)

print('Using CSV:', CITY_CSV)


Using CSV: /content/drive/MyDrive/ml/PROCESSED/matched/city_divided/shanghai_data.csv


In [11]:
# Load the CSV with pandas (robust for various environments)
df = pd.read_csv(CITY_CSV)

# Ensure datetime columns parsed
for col in ['receipt_time', 'sign_time']:
    if col in df.columns and not np.issubdtype(df[col].dtype, np.datetime64):
        df[col] = pd.to_datetime(df[col], format='%Y-%m-%d %H:%M:%S', errors='coerce')

# Derive or ensure 'ds' (MMDD) exists. If receipt_time contains full datetime, use it;
# otherwise rely on an existing 'ds' column in the CSV.
if 'ds' not in df.columns:
    if 'receipt_time' in df.columns:
        df['ds'] = df['receipt_time'].dt.strftime('%m%d')
    else:
        df['ds'] = pd.NA
# Normalize ds to string (zero-padded MMDD)
df['ds'] = df['ds'].astype(str)

# Basic cleaning: drop rows missing key timestamps or courier id
df = df.dropna(subset=['receipt_time', 'delivery_user_id'])
df['delivery_user_id'] = df['delivery_user_id'].astype(str)

# Compute eta_mins if present/needed
if 'sign_time' in df.columns:
    df['eta_mins'] = (df['sign_time'] - df['receipt_time']).dt.total_seconds() / 60

# Sort as requested: by delivery_user_id then ds then receipt_time then order_id (if exists)
sort_cols = ['delivery_user_id', 'ds', 'receipt_time']
if 'order_id' in df.columns:
    sort_cols.append('order_id')
df = df.sort_values(sort_cols).reset_index(drop=True)

print('Loaded rows:', len(df))
df.head(3)


Loaded rows: 34735


,order_id,from_dipan_id,delivery_user_id,poi_lng,poi_lat,aoi_id,typecode,receipt_time,receipt_lng,receipt_lat,sign_time,sign_lng,sign_lat,ds,city,eta_mins
0,04fc2f9b94c6de1069d525e259ca7d82,2cf0d24cee3c0ad128ad76019f018f9a,00fca617ad52d2deb9650342901a1940,1.056095e+07,-7.454289e+06,8cd1cc14a0e53f305b80dbae07983ada,203ac3454d75e02ebb0a3c6f51d735e4,2021-03-18 07:24:00,1.056254e+07,-7.452504e+06,2021-03-18 08:29:00,NaN,NaN,318,Shanghai,65.0
1,0a11f8e3fee958aa3df8e7ceab8a5146,2cf0d24cee3c0ad128ad76019f018f9a,00fca617ad52d2deb9650342901a1940,1.056086e+07,-7.454063e+06,d4a631a1f2165ab095adb674382b3eea,203ac3454d75e02ebb0a3c6f51d735e4,2021-03-18 07:24:00,1.056254e+07,-7.452508e+06,2021-03-18 08:47:00,NaN,NaN,318,Shanghai,83.0
2,c1039a5e963e50ac59b037f9f0c7a3ac,2cf0d24cee3c0ad128ad76019f018f9a,00fca617ad52d2deb9650342901a1940,1.056094e+07,-7.454282e+06,8cd1cc14a0e53f305b80dbae07983ada,203ac3454d75e02ebb0a3c6f51d735e4,2021-03-18 07:24:00,1.056252e+07,-7.452511e+06,2021-03-18 08:09:00,NaN,NaN,318,Shanghai,45.0


In [14]:
# Compute batch features: batch_size, batch_id, batch_rank_dispatch
# Definition: orders sharing (delivery_user_id, ds, receipt_time) are a batch — include ds to separate days
group_cols = ['delivery_user_id', 'ds', 'receipt_time']
if 'order_id' in df.columns:
    df['batch_size'] = df.groupby(group_cols)['order_id'].transform('count')
else:
    df['batch_size'] = df.groupby(group_cols).transform('size')
# batch_id: courier__ds__epoch(receipt_time) — ds ensures uniqueness when receipt_time lacks a date
df['batch_id'] = df['delivery_user_id'] + '__' + df['ds'].astype(str) + '__' + df['receipt_time'].astype('int64').floordiv(10**9).astype(str)
# dispatch rank: deterministic order within the simultaneous push — use order_id if available else row order
if 'order_id' in df.columns:
    df['batch_rank_dispatch'] = df.groupby(group_cols)['order_id'].cumcount()
else:
    df['batch_rank_dispatch'] = df.groupby(group_cols).cumcount()

# actual rank by sign_time (post-hoc), stored only for diagnostics if sign_time exists
if 'sign_time' in df.columns:
    df['batch_rank_actual'] = df.groupby(group_cols)['sign_time'].rank(method='first').astype('Int64') - 1
else:
    df['batch_rank_actual'] = pd.NA

print('Batch features computed. Example:')
print(display(df[['delivery_user_id','ds','receipt_time','batch_id','batch_size','batch_rank_dispatch','batch_rank_actual']].head(8)))


Batch features computed. Example:


,delivery_user_id,ds,receipt_time,batch_id,batch_size,batch_rank_dispatch,batch_rank_actual
0,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:24:00,00fca617ad52d2deb9650342901a1940__318__1616052240,4,0,1
1,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:24:00,00fca617ad52d2deb9650342901a1940__318__1616052240,4,1,3
2,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:24:00,00fca617ad52d2deb9650342901a1940__318__1616052240,4,2,0
3,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:24:00,00fca617ad52d2deb9650342901a1940__318__1616052240,4,3,2
4,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:25:00,00fca617ad52d2deb9650342901a1940__318__1616052300,6,0,5
5,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:25:00,00fca617ad52d2deb9650342901a1940__318__1616052300,6,1,0
6,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:25:00,00fca617ad52d2deb9650342901a1940__318__1616052300,6,2,1
7,00fca617ad52d2deb9650342901a1940,318,2021-03-18 07:25:00,00fca617ad52d2deb9650342901a1940__318__1616052300,6,3,2


None


In [18]:
# Check for dynamic pickup per day: for each courier, for each batch, did a next batch arrive while some
# orders in the current batch were still undelivered? (i.e., sign_time > next_batch_receipt_time)

# We need sign_time for this diagnostic; if absent, report and stop
if 'sign_time' not in df.columns:
    print('sign_time column missing — cannot compute dynamic pickup. Add sign_time to CSV and retry.')
else:
    # Analyse per-day (ds) so that batches are isolated by day
    ds_values = sorted(df['ds'].dropna().unique())
    if len(ds_values) == 0:
        ds_values = [None]
    for ds_val in ds_values:
        print('\n=== Day:', ds_val, '===')
        if ds_val is None:
            df_day = df.copy()
        else:
            df_day = df[df['ds'] == ds_val].copy()
        if df_day.empty:
            print('No data for this day.')
            continue

        # Build per-courier batch summary sorted by receipt_time for this day
        if 'order_id' in df_day.columns:
            batches = df_day.groupby(['delivery_user_id','batch_id'], as_index=False).agg({
                'receipt_time': 'first',
                'batch_size': 'first',
                'order_id': lambda s: list(s),
            })
        else:
            batches = df_day.groupby(['delivery_user_id','batch_id'], as_index=False).agg({
                'receipt_time': 'first',
                'batch_size': 'first',
            })
        batches = batches.sort_values(['delivery_user_id','receipt_time']).reset_index(drop=True)

        # Map next receipt_time per courier (shift) within the same day
        batches['next_batch_receipt'] = batches.groupby('delivery_user_id')['receipt_time'].shift(-1)

        # For each batch, count unfinished orders at next_batch_receipt
        def count_unfinished(row):
            if pd.isna(row['next_batch_receipt']):
                return 0
            mask = (df_day['batch_id'] == row['batch_id']) & (df_day['sign_time'] > row['next_batch_receipt'])
            return int(mask.sum())

        batches['unfinished_at_next'] = batches.apply(count_unfinished, axis=1)
        batches['dynamic_pickup_flag'] = (batches['unfinished_at_next'] > 0).astype(int)

        # Summary statistics for this day
        total_batches = len(batches)
        dyn_count = int(batches['dynamic_pickup_flag'].sum())
        if total_batches > 0:
            print(f'Total batches (detected): {total_batches:,}')
            print(f'Batches with dynamic pickup (new batch arrived while previous had unfinished deliveries): {dyn_count:,} ({dyn_count/total_batches*100:.2f} %)')
        else:
            print('No batches detected for this day.')

        # Per-courier dynamic pickup analysis for this day
        print('\n--- Per-courier Dynamic Pickup for Day:', ds_val, '---')
        courier_summary = batches.groupby('delivery_user_id').agg(
            total_batches_courier=('batch_id', 'count'),
            dynamic_pickup_batches_courier=('dynamic_pickup_flag', 'sum')
        ).reset_index()
        courier_summary['pct_dynamic_pickup'] = (
            courier_summary['dynamic_pickup_batches_courier'] /
            courier_summary['total_batches_courier'] * 100
        )
        # Display top 10 couriers by percentage of dynamic pickups
        display(courier_summary[courier_summary['total_batches_courier'] > 0]
                .sort_values('pct_dynamic_pickup', ascending=False).head(10))
        if len(courier_summary) > 10:
            print(f'... and {len(courier_summary) - 10} more couriers (showing top 10 by percentage).')

        # Distribution by batch size
        if total_batches > 0:
            size_tab = batches.groupby('batch_size')['dynamic_pickup_flag'].agg(['count','sum']).reset_index()
            size_tab['pct_dynamic'] = size_tab['sum'] / size_tab['count'] * 100
            display(size_tab.sort_values('batch_size').head(20))

        # Show example cases where dynamic pickup happened
        examples = batches[batches['dynamic_pickup_flag'] == 1].head(10)
        if not examples.empty:
            for _, r in examples.iterrows():
                print('Example courier:', r['delivery_user_id'], 'receipt_time:', r['receipt_time'])
                display(df_day[df_day['batch_id'] == r['batch_id']][['order_id','receipt_time','sign_time','batch_rank_dispatch','batch_rank_actual']].sort_values('batch_rank_dispatch'))
        else:
            print('No dynamic pickup examples found for this day.')

        # Optional: save batch-level summary for this day
        OUT = './batch_dynamics_summary_' + CITY.lower() + '_' + str(ds_val) + '.csv'
        batches.to_csv(OUT, index=False)
        print('Saved batch summary to', OUT)


=== Day: 318 ===
Total batches (detected): 903
Batches with dynamic pickup (new batch arrived while previous had unfinished deliveries): 631 (69.88 %)

--- Per-courier Dynamic Pickup for Day: 318 ---


,delivery_user_id,total_batches_courier,dynamic_pickup_batches_courier,pct_dynamic_pickup
19,2cd6a9a317e3cc0dc8aae36443f5d806,16,14,87.500000
20,2eac82477425ea9e3b76985a3e8a1715,21,18,85.714286
2,0624b34f057ea763a1af4721b89d7b7c,7,6,85.714286
55,a953e787e74cc318462aa121e52d9896,21,18,85.714286
42,6caec65b164b4e29a27509a10c3736a6,6,5,83.333333
56,b2189fdf3fa9e164ac2200e0b0d80af4,12,10,83.333333
11,156ba74f4cbb5799f74ba615fa586104,17,14,82.352941
63,d1fda897c90233353f906db959a62b17,17,14,82.352941
40,6a76231746403d5448f6a5217034980d,16,13,81.250000
18,2bda9e538a98b75de75a2df2114c5f9b,16,13,81.250000


... and 69 more couriers (showing top 10 by percentage).


,batch_size,count,sum,pct_dynamic
0,1,377,248,65.782493
1,2,216,159,73.611111
2,3,137,102,74.452555
3,4,72,53,73.611111
4,5,42,29,69.047619
5,6,27,15,55.555556
6,7,16,14,87.500000
7,8,9,5,55.555556
8,9,5,4,80.000000
9,11,1,1,100.000000


Example courier: 00fca617ad52d2deb9650342901a1940 receipt_time: 2021-03-18 07:24:00


...

Example courier: 00fca617ad52d2deb9650342901a1940 receipt_time: 2021-04-01 16:26:00


,order_id,receipt_time,sign_time,batch_rank_dispatch,batch_rank_actual
540,70f84e4e86921598e7497ff6e2780fa0,2021-04-01 16:26:00,2021-04-01 19:47:00,0,5
541,8d6246c6050cb1d77392c47654f1e99c,2021-04-01 16:26:00,2021-04-01 18:32:00,1,0
542,b3feb62599df7b3f59071f9a8c339c71,2021-04-01 16:26:00,2021-04-01 18:43:00,2,1
543,bfacb5e8673ff52bb1bc3de16cdf1cfa,2021-04-01 16:26:00,2021-04-01 19:08:00,3,4
544,e4f07c20bfc03a3b79852780481cd84b,2021-04-01 16:26:00,2021-04-01 18:50:00,4,2
545,ffba1566b3eb44f0b809f84c78664e6e,2021-04-01 16:26:00,2021-04-01 19:02:00,5,3


Example courier: 00fca617ad52d2deb9650342901a1940 receipt_time: 2021-04-01 16:27:00


,order_id,receipt_time,sign_time,batch_rank_dispatch,batch_rank_actual
546,e81549607af585412ae0282285d91096,2021-04-01 16:27:00,2021-04-01 20:00:00,0,0


Saved batch summary to ./batch_dynamics_summary_shanghai_401.csv


In [ ]:
# Optional: save batch-level summary for further analysis
OUT = './batch_dynamics_summary_' + CITY.lower() + '.csv'
batches.to_csv(OUT, index=False)
print('Saved batch summary to', OUT)


## Notes
- This notebook treats orders with identical `receipt_time` (same courier) as part of the same batch, matching the LaDe pipeline's definition.
- Dynamic pickup is detected when a later batch's `receipt_time` occurs while some orders in the earlier batch remain unsigned (`sign_time > next_batch_receipt_time`).
- If `sign_time` is missing or inaccurate, consider using courier GPS traces or delivery status logs to refine the overlap definition.

If you want, I can run this notebook for a specific city file in your environment, or extend it to visualise temporal overlap per courier.